# Where is the structure: in the neurons, or in the areas?

The three clustering tests of the shape-metrics paper, applied to the non-human
primate recordings of Siegel, Buschman & Miller (2015). They differ only in what
the points being clustered *are*:

1. **is each area categorical?** -- one $z$ per area, its own neurons against a
   Gaussian matched to that area (figure 2b);
2. **neurons pooled across areas** -- every neuron of every area in one cloud,
   the analysis figure 1 uses to show that pooling answers a different question
   than it appears to;
3. **the areas themselves** -- whole populations compared with the Procrustes
   shape distance, then clustered in the resulting region space (figure 2f).

They are easily conflated and can disagree. A pooled cloud is a mixture over
areas, and a mixture is lumpy whenever areas differ from one another, whether or
not any area contains cell types -- so test 2 can be positive with tests 1 and 3
both flat. Only test 3 speaks to whether the areas form types.

The statistics, the nulls and the plotting style are figure 2's, so the numbers
are comparable with the IBL Brainwide Map in `Posani/notebook.ipynb`. Nothing is
re-implemented: the clustering machinery comes from `shapemetrics/`, the house
style from `Posani/code/plotting.py`.

**Two things are needed to port figure 2b to this dataset, and section 1 shows
why.** Figure 2's IBL analysis admits only neurons whose encoding model beats a
variable-agnostic null ($\Delta R^2 > 0.015$). These recordings come with no such
filter, and most of the units are not reliable: the median split-half correlation
across the ten folds is $-0.06$, and 78% of units fall below $r = 0.1$. The
clustering pipeline standardises each neuron across conditions, which divides a
near-flat noisy curve by its own tiny standard deviation and inflates it into a
full-amplitude one -- and a cloud of those is close to uniform on the sphere,
which is maximally *un*clusterable. So the reliability criterion is the first
requirement. The second is that the Gaussian null must be drawn in whatever
representation the test receives, which for a standardised test means drawing it
from standardised curves.

In [ ]:
import os
os.environ["TQDM_DISABLE"] = "1"            # the vendored pipeline is chatty

import pickle
from pathlib import Path

from shapemetrics import paths

paths.set_figure("Figure2")
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

import shapemetrics as sm                                              # noqa: E402
from shapemetrics import plotting

OUT = paths.results()
OUT.mkdir(exist_ok=True)
plt.rcParams.update({"svg.fonttype": "none", "text.usetex": False})

AREAS = ["PFC", "FEF", "LIP", "Parietal", "IT", "MT", "V4"]

MIN_R = 0.2                                    # split-half reliability criterion
MIN_NEURONS, N_SUB, N_PCS, N_REPEATS = 80, 80, 5, 20
KLIM, NINIT, N_DRAWS = (2, 11), 10, 25         # figure 2b's settings
N_POOL, N_NULL_POOL = 2000, 500                # figure 1's pooled-neuron test
N_NULL_REGIONS, EMB_DIM = 500, 5               # figure 2f's settings

# `standardise` works on `neurons x conditions`; `z_over_conditions` on the
# `folds x conditions x neurons` stacks the reliability is measured from -- the
# axis matters, and getting it wrong silently inflates the correlations
standardise = lambda M: (M - M.mean(1, keepdims=True)) / (M.std(1, keepdims=True) + 1e-12)
z_over_conditions = lambda M: (M - M.mean(1, keepdims=True)) / (M.std(1, keepdims=True) + 1e-12)

## The data

Ten folds, each holding two independent halves of the trials, and within each an
area $\times$ monkey population of `63 conditions x neurons` (4 colours $\times$ 4
directions $\times$ 4 contextual cues, one condition absent).

**No cross-validation in the summaries.** The other analyses of this dataset
compare one partition against another, which is right when the question is
whether a distance survives independent trials. These tests ask how lumpy a cloud
is, and a split-half estimate only adds noise there -- which depresses the
observed silhouette while the Gaussian null, drawn from the estimated covariance,
is not depressed the same way. So each area is summarised once, from all its
trials. The folds are used for one thing only: to measure each neuron's
reliability, by correlating its tuning across the two halves.

**The two monkeys are pooled into one population per area.** The question is
whether *areas* differ, so an area is one population and not two, and the 63
conditions are shared across subjects. This is also what makes the sensory end of
the hierarchy available: IT, MT and V4 clear the unit floor only once both
monkeys are counted. The cost is that each area's population mixes two brains.

In [ ]:
folds = pickle.load(open("data/Siegel_10_folds_avg_stim_netrep.pkl", "rb"))
units = list(folds[0][0])
area_of, monkey_of = (lambda k: k.rsplit("_", 1)[0]), (lambda k: k.rsplit("_", 1)[1])

# units that never vary carry no tuning and cannot be standardised
alive = {k: np.mean([p[k] for f in folds for p in f], 0).std(0) > 0 for k in units}

# split-half reliability: correlate each neuron's tuning across the two halves
reliability, curves = {}, {}
for k in units:
    A = np.stack([f[0][k] for f in folds])[:, :, alive[k]]
    B = np.stack([f[1][k] for f in folds])[:, :, alive[k]]
    # correlate the two halves ACROSS CONDITIONS, per neuron, per fold
    za, zb = z_over_conditions(A), z_over_conditions(B)
    reliability[k] = np.nanmean((za * zb).mean(1), 0)
    curves[k] = np.mean([p[k] for f in folds for p in f], 0)[:, alive[k]]

rel = {a: np.concatenate([reliability[k] for k in units if area_of(k) == a]) for a in AREAS}
raw = {a: np.hstack([curves[k] for k in units if area_of(k) == a]) for a in AREAS}

allr = np.concatenate([rel[a] for a in AREAS])
print(f"{len(allr)} units; split-half reliability median {np.median(allr):+.3f}, "
      f"{(allr < 0.1).mean():.0%} below 0.1\n")
print(f"{'area':<9}{'all':>7}{'r > ' + str(MIN_R):>10}")
for a in AREAS:
    print(f"{a:<9}{len(rel[a]):>7}{(rel[a] > MIN_R).sum():>10}")

# the neuron-level tests run on reliable units, standardised
X = {a: standardise(raw[a][:, rel[a] > MIN_R].T) for a in AREAS}     # neurons x conditions
print(f"\nkept {sum(v.shape[0] for v in X.values())} units at r > {MIN_R}")

## 1. Why a reliability criterion, and why not PCA instead

Three things checked rather than asserted.

**The deficit is entirely a reliability effect.** Run on all units, the pooled
test comes out at $z = -4.9$: the data are *less* clusterable than a Gaussian
matched to them, which is not a statement about categoricality at all. Restrict
by reliability and it moves monotonically into the ordinary range.

**Centring instead of standardising does not fix it.** It swaps one artefact for
another: with amplitude retained, $k$-means finds firing-rate outliers, splitting
a few dozen loud neurons off from everything else, and the silhouette that
produces has nothing to do with tuning shape. Note that centring and then scaling
each curve to unit norm is *exactly* what standardising does, so these are the two
ends of one axis rather than independent options.

**No PCA denoising is applied**, and the check below is why it is not needed:
fitting the leading components on one trial half and scoring them on the other,
the cross-validated $R^2$ is still climbing at full rank, so among reliable units
every component carries signal and truncating would discard it rather than remove
noise. Selection, not projection, is what this dataset needs.

In [ ]:
def pooled_test(A, n_draw, rng):
    """Figure 1's pooled-neuron test on a `neurons x conditions` matrix."""
    obs = sm.pipeline_silhouette(A, k_lim=KLIM, n_init=NINIT)
    null = np.array([sm.pipeline_silhouette(sm.curve_gaussian_null(A, rng),
                                            k_lim=KLIM, n_init=NINIT)
                     for _ in range(n_draw)])
    return obs, null, (obs - null.mean()) / (null.std() + 1e-12)


rng = np.random.default_rng(0)
allC = np.hstack([raw[a] for a in AREAS])                    # conditions x neurons
print("pooled-neuron test as the reliability criterion is tightened:")
for thr in (-np.inf, 0.0, 0.1, 0.2, 0.3):
    idx = np.flatnonzero(allr > thr)
    take = rng.choice(idx, min(N_POOL, len(idx)), replace=False)
    o, n, z = pooled_test(standardise(allC[:, take].T), 15, rng)
    lab = "all units" if thr == -np.inf else f"r > {thr:.1f}"
    print(f"  {lab:<12} n = {len(take):>4}   obs {o:.4f}   null {n.mean():.4f}   z = {z:+6.2f}")

print("\ncentring instead of standardising (r > 0.2), and what its clusters are:")
from sklearn.cluster import KMeans
Cc = (allC[:, allr > MIN_R] - allC[:, allr > MIN_R].mean(0)).T
o, n, z = pooled_test(Cc, 10, rng)      # pipeline_silhouette standardises; pass scale=False
o = sm.pipeline_silhouette(Cc, k_lim=KLIM, n_init=NINIT, scale=False)
n = np.array([sm.pipeline_silhouette(sm.curve_gaussian_null(Cc, rng), k_lim=KLIM,
                                     n_init=NINIT, scale=False) for _ in range(10)])
lab2 = KMeans(2, n_init=10, random_state=0).fit_predict(sm.clustering_space(Cc, scale=False))
amp = np.linalg.norm(Cc, axis=1)
print(f"  obs {o:.4f}  null {n.mean():.4f}  z = {(o - n.mean()) / n.std():+.1f}")
print(f"  k = 2 gives clusters of {sorted(np.bincount(lab2).tolist())}, "
      f"median amplitude {[f'{np.median(amp[lab2 == i]):.0f}' for i in range(2)]}")

print("\ncross-validated PCA rank among reliable units:")
h = [np.hstack([np.mean([f[i][k][:, alive[k]] for f in folds], 0) for k in units])[:, allr > MIN_R]
     for i in (0, 1)]
A_, B_ = h[0] - h[0].mean(0), h[1] - h[1].mean(0)
U, S, Vt = np.linalg.svd(A_, full_matrices=False)
for m in (5, 20, min(A_.shape) - 1):
    r2 = 1 - (((B_ - (U[:, :m] * S[:m]) @ Vt[:m]) ** 2).sum() / (B_ ** 2).sum())
    print(f"  {m:>3} components: cross-validated R2 = {r2:.3f}")

## 2. Is each area categorical on its own?

Figure 2b. Each area's reliable neurons go through Posani et al.'s pipeline --
standardise each neuron across the 63 conditions, PCA to 90% of the variance,
sweep $k$, keep the best mean silhouette -- against `N_DRAWS` Gaussians matched to
*that* area and passed through the identical pipeline. One $z$ per area, and the
histogram of those seven values is the first panel of the figure.

In [ ]:
f = OUT / "region_categoricality.npz"
if f.exists():
    z_cat = np.load(f)["z"]
else:
    z_cat = []
    for a in AREAS:
        rng = np.random.default_rng(0)
        z_cat.append(pooled_test(X[a], N_DRAWS, rng)[2])
    z_cat = np.array(z_cat)
    np.savez(f, z=z_cat, regions=np.array(AREAS))

print(f"{(np.abs(z_cat) > 1.96).sum()}/{len(z_cat)} areas past |z| = 1.96, "
      f"median z = {np.median(z_cat):+.2f}, "
      f"range {z_cat.min():+.2f} to {z_cat.max():+.2f}\n")
for a, z_ in sorted(zip(AREAS, z_cat), key=lambda t: -t[1]):
    print(f"  {a:<9} n = {X[a].shape[0]:>4}   z = {z_:+.2f}")
n_kept = np.array([X[a].shape[0] for a in AREAS])
print(f"\nspearman(z, units per area) = {stats.spearmanr(n_kept, z_cat).statistic:+.2f} "
      f"(p = {stats.spearmanr(n_kept, z_cat).pvalue:.3f})")

## 3. Neurons pooled across areas

The same statistic and the same null, on every reliable neuron of every area at
once.

This is the test figure 1 is about. A pooled cloud is a mixture over areas, and a
mixture is lumpy whenever the areas differ from one another -- whether or not any
area contains cell types. In the figure 1 simulations, pooling was positive in
every scenario, including fields built with no region types in them at all. So a
positive result here licenses a statement about the neurons and none at all about
the areas; section 4 is what answers that.

Silhouette is $O(n^2)$, so the pooled cloud is subsampled to `N_POOL` neurons.

In [ ]:
f = OUT / "pooled_neurons.npz"
if f.exists():
    dd = np.load(f)
    pool_obs, pool_null = float(dd["obs"]), dd["null"]
else:
    rng = np.random.default_rng(0)
    allX = np.vstack([X[a] for a in AREAS])
    Xp = allX[rng.choice(len(allX), min(N_POOL, len(allX)), replace=False)]
    pool_obs, pool_null, _ = pooled_test(Xp, N_NULL_POOL, rng)
    np.savez(f, obs=pool_obs, null=pool_null)

pool_z = (pool_obs - pool_null.mean()) / pool_null.std()
pool_p = (np.sum(pool_null >= pool_obs) + 1) / (len(pool_null) + 1)
print(f"pooled neurons: silhouette {pool_obs:.4f} against {pool_null.mean():.4f} "
      f"+/- {pool_null.std():.4f}   z = {pool_z:+.2f}, p = {pool_p:.3f}")

## 4. Do the areas themselves fall into types?

Figure 2f, and the only one of the three that speaks to the areas.

This test runs on **all** units, not only the reliable ones. It does not
standardise per neuron -- `preprocess` reduces each population to its leading
components and normalises the whole matrix -- so it never divides a flat curve by
its own noise, and the failure that motivates the reliability criterion above
cannot occur here. Unreliable units add roughly isotropic noise, which inflates
all the distances together rather than any one of them. Keeping every unit is
also what lets all seven areas be subsampled to the same `N_SUB`; the reliable
subsets of Parietal and IT are smaller than that. The version restricted to
reliable units is reported underneath.

Each area becomes one point cloud, compared with the Procrustes distance; every
area is subsampled to `N_SUB` units and the matrix averaged over `N_REPEATS`
draws, so that area size cannot drive the distances. The areas are then embedded
by classical MDS and scored by the best silhouette over $k$, against Gaussians
matched to that embedding's mean and covariance. Silhouette cannot score the
one-cluster hypothesis, which is why "no types" has to be simulated rather than
evaluated.

In [ ]:
def region_test(pops, names, n_sub, seed=0):
    rng = np.random.default_rng(seed)
    D = np.zeros((len(names), len(names)))
    for _ in range(N_REPEATS):
        P = [sm.preprocess(pops[a][:, rng.choice(pops[a].shape[1], n_sub, replace=False)],
                           N_PCS) for a in names]
        D += sm.pairwise(P)
    D /= N_REPEATS
    E = sm.classical_mds(D, min(EMB_DIM, len(names) - 1))
    ks = np.arange(2, len(names))
    obs = sm.silhouette_sweep(E, ks)
    nl = sm.gaussian_null(E, ks, n_draw=N_NULL_REGIONS).max(1)
    return dict(D=D, sil=obs.max(), best_k=int(ks[obs.argmax()]), null=nl,
                z=(obs.max() - nl.mean()) / nl.std(),
                p=(np.sum(nl >= obs.max()) + 1) / (len(nl) + 1))


rel_pops = {a: raw[a][:, rel[a] > MIN_R] for a in AREAS}
N_SUB_REL = min(v.shape[1] for v in rel_pops.values())     # capped by the smallest

r_all = region_test(raw, AREAS, N_SUB)                     # every unit
r_rel = region_test(rel_pops, AREAS, N_SUB_REL)            # reliable units only

iu = np.triu_indices(len(AREAS), 1)
for lab, res, n_sub, pops in (("all units", r_all, N_SUB, raw),
                              (f"r > {MIN_R}", r_rel, N_SUB_REL, rel_pops)):
    D_ = res["D"]
    print(f"{lab:<10} {n_sub:>3} per area:  distances {D_[iu].mean():.3f} "
          f"({D_[iu].min():.3f} to {D_[iu].max():.3f})   best k = {res['best_k']}, "
          f"silhouette = {res['sil']:.4f}, null {res['null'].mean():.4f} "
          f"+/- {res['null'].std():.4f}, z = {res['z']:+.2f}, p = {res['p']:.3f}")

## 5. The figure

The three tests side by side, in figure 2's style: the spread of per-area
categoricality, then the two that are most often confused -- neurons pooled
across areas, and the areas themselves.

In [ ]:
PANEL = 2.55
fig, axes = plt.subplots(1, 4, figsize=(4 * PANEL, 1.15 * PANEL))

plotting.categoricality_hist(axes[0], z_cat)
axes[0].set_title("is each area\ncategorical?")

plotting.null_hist(axes[1], pool_obs, pool_null, "mean silhouette",
                   label_null="one blob", label_obs="neurons", p=pool_p)
axes[1].set_title("neurons pooled\nacross areas")

# the areas, both ways: every unit, and reliable units only
plotting.null_hist(axes[2], r_all["sil"], r_all["null"], "best silhouette\nbetween areas",
                   label_null="one blob", label_obs="areas", p=r_all["p"])
axes[2].set_title("areas in shape space\n(all units)")

plotting.null_hist(axes[3], r_rel["sil"], r_rel["null"], "best silhouette\nbetween areas",
                   label_null="one blob", label_obs="areas", p=r_rel["p"])
axes[3].set_title(f"areas in shape space\n(r > {MIN_R})")

for ax, letter in zip(axes, "abcd"):
    plotting.panel_letter(ax, letter, dx=-22)
plotting.typeset(fig)
fig.tight_layout(w_pad=0.9)
plotting.save(fig, "siegel_region_clustering", folder="results")

## What it says

To be filled in from the numbers above.